In [ ]:
!pip install requests google-auth

In [27]:
import requests
import google.auth
from google.auth.transport.requests import Request
import json

def stream_assist_to_agent(
    project_id: str,
    location: str,
    engine_id: str,
    query: str,
    agent_id: str
) -> None:
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    credentials, _ = google.auth.default(scopes=scopes)
    credentials.refresh(Request())
    access_token = credentials.token
    
    # 위치에 따른 API 엔드포인트 설정 (global인 경우 기본값 사용)
    if location.lower() == "global":
        endpoint = "discoveryengine.googleapis.com"
    else:
        endpoint = f"{location}-discoveryengine.googleapis.com"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    url = (
        f"https://{endpoint}/v1alpha/projects/{project_id}"
        f"/locations/{location}/collections/default_collection"
        f"/engines/{engine_id}/assistants/default_assistant:streamAssist"
    )

    payload = {
        "query": {
            "text": query
        },
        "agentsSpec": {
            "agentSpecs": [{"agentId": agent_id}]
        }
    }
    
    print(f"질문: {query}\n")
    
    with requests.post(url, headers=headers, json=payload, stream=True) as response:
        if response.status_code != 200:
            print(f"API 호출 오류 ({response.status_code}): {response.text}")
            return

        # JSON 파싱을 위한 임시 버퍼
        buffer = ""
        was_thought_previously = False

        for chunk in response.iter_content(chunk_size=1024, decode_unicode=True):
            if not chunk:
                continue
                
            buffer += chunk
            
            # 버퍼 내에서 온전한 JSON 객체 하나를 찾기 위해
            # 중괄호의 짝을 맞춰서 추출합니다.
            while True:
                try:
                    # 버퍼에서 첫 번째 '{' 의 위치
                    start_idx = buffer.find('{')
                    if start_idx == -1:
                        break # 열린 중괄호가 없으면 다음 청크 대기

                    # 중괄호 짝을 맞춰가며 하나의 완전한 객체 문자열 추출
                    open_braces = 0
                    end_idx = -1
                    
                    for i in range(start_idx, len(buffer)):
                        if buffer[i] == '{':
                            open_braces += 1
                        elif buffer[i] == '}':
                            open_braces -= 1
                            if open_braces == 0:
                                end_idx = i
                                break
                    
                    # 짝이 맞는 닫힌 중괄호를 찾지 못했다면 아직 청크가 덜 온 것
                    if end_idx == -1:
                        break
                        
                    # 온전한 JSON 문자열 하나를 추출하고 버퍼에서 제거
                    json_str = buffer[start_idx:end_idx+1]
                    buffer = buffer[end_idx+1:]
                    
                    # 추출한 문자열을 JSON 객체로 파싱
                    packet = json.loads(json_str)
                    
                    # 4. 내용 추출 및 출력 포맷팅
                    replies = packet.get("answer", {}).get("replies", [])
                    for reply in replies:
                        content = reply.get("groundedContent", {}).get("content", {})
                        text = content.get("text", "")
                        is_thought = content.get("thought", False)
                        
                        if text:
                            if is_thought:
                                # thought가 시작될 때 또는 계속 이어질 때 회색으로 표시(콘솔 환경)
                                if not was_thought_previously:
                                    print("\n\n[모델의 사고 과정]")
                                print(f"\033[90m{text}\033[0m", end="", flush=True) # 90m은 콘솔 색상(회색)
                                was_thought_previously = True
                            else:
                                # 본문이 시작될 때 구분선 추가
                                if was_thought_previously:
                                    print("\n\n[실제 답변]\n")
                                print(text, end="", flush=True)
                                was_thought_previously = False
                                
                except json.JSONDecodeError:
                    # 예상치 못한 에러 발생 시 루프를 빠져나가 다음 청크 수신
                    break

    print("\n\n--- 스트리밍 종료 ---")

In [28]:
PROJECT_ID = "sandbox-373102"
LOCATION = "global" # 또는 "us", "eu"
APP_ID = "test2_1775626181600" # Gemini Enterprise 앱(엔진) ID
QUERY_TEXT = "Gemini Enterprise의 주요 기능에 대해 설명해줘."
AGENT_ID = "9437998303491634412"

stream_assist_to_agent(
    project_id=PROJECT_ID,
    location=LOCATION,
    engine_id=APP_ID,
    query=QUERY_TEXT,
    agent_id=AGENT_ID
)

질문: Gemini Enterprise의 주요 기능에 대해 설명해줘.



[모델의 사고 과정]
**Clarifying Search Tool Usage**
**Prioritizing Internal Search**
**Confirming Tool Availability**
**Exploring Tool Accessibility**
**Executing Search Plan Directly**
**Confirming Search Capability**
**Resolving Execution Conflict**
**Clarifying Search Capability**
**Answering Query Directly**
**Answering Query Directly**
**Answering Query Directly**
**Answering Query Directly**


[실제 답변]

Gemini Enterprise는 조직의 생산성을 극대화하기 위해 Google Workspace에 최첨단 AI 기술을 통합한 포괄적인 비즈니스 솔루션입니다. 저 Gemini Enterprise는 사용자가 일상적으로 사용하는 도구 내에서 직접 창의적인 작업을 돕고, 업무 효율을 높이는 다양한 기능을 제공합니다.

---

### 🚀 주요 기능 안내

| 기능 카테고리 | 상세 설명 |
|---|---|
| **Google Workspace 통합** | Gmail, 문서(Docs), 스프레드시트(Sheets), 프레젠테이션(Slides)에서 직접 텍스트 작성, 요약, 데이터 정리 및 이미지 생성을 지원합니다. |
| **고급 AI 대화 (Gemini 앱)** | 엔터프라이즈급 데이터 보호가 적용된 환경에서 최신 모델인 Gemini 1.5 Pro를 활용하여 복잡한 질문에 답하고 긴 문서를 분석할 수 있습니다. |
| **AI 화상 회의 (Meet)** | 실시간 다국어 자막 번역, 회의 자동 요약(Take notes for me), 영상 및 음질 보정 기능을 통해 협업의 질을 높입